# 3.2 Autonomous Flight with Vision - Object Recognition

## Overview

The course workflow is: using "natural language commands", the drone operates under LLM control -- taking off, performing "visual perception", then making decisions based on perception results and executing actions.

"Perception" here is implemented through the LLM's multimodal capabilities, leveraging its image understanding.

Implementation is divided into two levels:

1. This lesson: The LLM can identify targets, but we need to tell the LLM the target's position, then test the LLM's reasoning capabilities.

2. Next lesson: The LLM needs to both identify targets AND locate their positions (object detection), then execute the target approach task.

In [ ]:
import sys
sys.path.append('..')
import airsim
import time

client = airsim.MultirotorClient()
client.confirmConnection()
client.enableApiControl(True)
client.armDisarm(True)

state = client.getMultirotorState()
pos = state.kinematics_estimated.position
print("Takeoff:", pos.x_val, pos.y_val, pos.z_val)


In [ ]:
client = airsim.MultirotorClient()
client.confirmConnection()
client.enableApiControl(True)
client.armDisarm(True)

# Takeoff
client.takeoffAsync().join()
time.sleep(1)

# Safe altitude
client.moveToZAsync(-3, 1).join()
time.sleep(1)

# Fly to the waypoint (point B)
target_x = 9.0
target_y = 16.5
target_z = -1.0

client.moveToPositionAsync(target_x, target_y, target_z, 2).join()
time.sleep(2)

# Hover
client.hoverAsync().join()

print("Arrived above point B")

## Adding Vision to the Drone

Based on the airsim_wrapper, we add the following function:

look()

Capability: Captures an image from the drone's front camera and returns a list of detected objects.

Specifically, the drone captures a camera image, which is then sent to the multimodal LLM for image understanding:

``` python
def look(self):
    """
    Get the front camera image and return a list of main objects in the image
    :return: string, object names separated by commas
    """
    # Step 1: Read camera image (already RGB)
    rgb_image = self.get_image()

    # Convert to Base64 PNG image
    base64_str = self.cv2_to_base64(rgb_image, ".png")  # png or '.jpg'

    # Step 2: Image understanding
    # Image input:
    response = self.llm_client.chat.completions.create(
        model="moonshotai/Kimi-K2.6",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "What objects are in this image? Just give me the names. Only list common, clearly visible objects. Separate multiple names with commas."},
                    {
                        "type": "image_url",
                        "image_url": {
                            # Use Base64-encoded local image, note img/png or img/jpg format
                            "url": f"data:image/png;base64,{base64_str}"
                        }
                    },
                ],
            }
        ],
        temperature=0.01
    )

    content = response.choices[0].message.content
    return content

```

## Prompt Engineering

In [ ]:
# Write knowledge base prompt file aisim_lession32.txt
kg_promot_file = "prompts/aisim_lession32.txt"

kg_prompt = """
Imagine you are helping me interact with the AirSim drone simulator. At any given point in time, you have the following capabilities, each identified by a unique label. You also need to output code for certain requests.

Question: You can ask me a clarification question, as long as you clearly identify it as a "Question".
Code: Output code commands that achieve the desired goal.
Reason: After outputting code, you should explain why you did it that way.

The simulator contains a drone and several objects. No objects other than the drone can move. In the code, we can use the following commands. You must not use any other hypothetical functions.

aw.takeoff() - Takes off the drone.
aw.land() - Lands the drone.

aw.get_drone_position() - Returns the drone's current position and heading angle as a list of 4 floats corresponding to (X, Y, Z, angle) coordinates.
aw.get_position(object_name): Takes a string as input indicating the name of an object of interest, and returns a vector of 4 floats indicating its (X, Y, Z, angle) coordinates.
aw.fly_to(position): Takes a vector of 4 floats as input indicating (X, Y, Z, angle) coordinates, and commands the drone to fly to that position along the specified angle.
aw.fly_path(positions): Takes a list of (X, Y, Z, angle) position coordinates representing waypoints along a path, and the drone flies along these waypoints.
aw.look_at(angle): Takes an angle as input and rotates the drone to face that direction.



aw.look(): Look around and get a list of objects visible in the current camera view, returned as comma-separated names.
aw.look_at(angle): Takes an angle as input and rotates the drone to face that direction.

The simulator world contains some items such as mirror, chair, rubber duck, coconut water, cola, and orchid.
You can use the get_position(object_name) function to get their positions.
For example, if you want to know the position of "chair", you can use:
aw.get_position('chair')

When generating code, you do not need to worry about importing aw; it is already declared in the environment.

"""

pt_file = open(kg_promot_file, "w", encoding="utf-8")
pt_file.write(kg_prompt)
pt_file.close()

## Setting Camera Resolution

AirSim's default camera resolution is only 256x144. This is too low for image recognition.

For our application, we need to modify the resolution to 720p (1280x720) for better results.

You can modify the resolution through the configuration file settings.json:

``` json
{
    "SettingsVersion": 1.2,
    "CameraDefaults": {
        "CaptureSettings": [
            {
                "ImageType": 0,
                "Width": 1280,
                "Height": 720,
                "FOV_Degrees": 90,
                "AutoExposureSpeed": 100,
                "MotionBlurAmount": 0
            }
        ]
    }
}
```

Parameter note: "SimMode": "ComputerVision"

## Execution Strategy

Because multimodal LLM outputs have significant non-determinism, if we let the language model directly execute actions based on multimodal LLM results, it may not be reliable.

Therefore, in the multimodal workflow, we use a "human-in-the-loop" approach: the LLM generates code, we confirm it, then execute. This gives us step-by-step control over the LLM's task execution.

That is, we call the LLM to generate code but don't execute it automatically:
```python
python_code = my_agent.process(command, False) # Don't execute
```

We execute only after manual confirmation.

In [ ]:
!pip install flask supervision 

## Start the Simulation

Run the .bat file to start the simulation environment.

Content:

House.exe -windowed -ResX=1280 -ResY=720 -ExecCmds="DisableAllScreenMessages"

In [ ]:
# Create an AirSim wrapper instance for execution
import airsim_wrapper
aw = airsim_wrapper.AirSimWrapper()

In [ ]:
# Create the LLM agent
import airsim_agent
my_agent = airsim_agent.AirSimAgent(knowledge_prompt="prompts/aisim_lession32.txt")

In [ ]:
command = "Take off"
ret = my_agent.process(command, False) # Generate code but don't execute
print("ret:", ret)

In [ ]:
# Manual execution -------

aw.takeoff()

In [ ]:
command = "Look around and tell me what objects are in the environment"
ret = my_agent.process(command, False) # Generate code but don't execute
print("ret:", ret)

In [ ]:
# Manual execution -------

objects = aw.look()
print(objects)

In [ ]:
command = """
Good, the environment contains: mirror, chair, rubber duck, coconut water, cola, orchid.
Now, fly to the cola.
"""
ret = my_agent.process(command, False) # Don't execute
print("ret:", ret)

In [ ]:
command = """
The environment contains the following objects: mirror, chair, rubber duck, coconut water, cola, orchid.
Now use the get_position function to get the cola's position.
"""
ret = my_agent.process(command, False) # Don't execute
print("ret:", ret)

In [ ]:
command = """
Good, now fly to the cola and hover in front of it.
"""
ret = my_agent.process(command, False) # Don't execute
print("ret:\n", ret)

In [ ]:
command = """
Can you simplify that? Don't overthink it. Just fly to a position 3 meters in front of the cola on the X axis.
"""
ret = my_agent.process(command, False) # Don't execute
print("ret:\n", ret)

In [ ]:
# Manual execution -------

# Get cola position
cola_pos = aw.get_position('cola')
# Calculate target position: 3 meters in front on X axis
target_pos = [cola_pos[0] - 3, cola_pos[1], cola_pos[2], cola_pos[3]]
# Fly drone to target position
aw.fly_to(target_pos)

In [ ]:
command = """
Can you see the cola from the drone's current position?
"""
ret = my_agent.process(command, False) # Don't execute
print("ret:\n", ret)

In [ ]:
command = """
The environment has a mirror. Fly in front of the mirror and look at it.
"""
ret = my_agent.process(command, False) # Don't execute
print("ret:\n", ret)

In [ ]:
# Manual execution -------

# get position
mirror_position = aw.get_position('')

# calculate before targetposition, false Xdistance1 before 
target_x = mirror_position[0] - 2
target_position = [target_x, mirror_position[1], mirror_position[2], mirror_position[3]]

# let dronetargetposition
aw.fly_to(target_position)

# let drone toward angle ( this inside false angle not) 
aw.look_at(0)

# simulation, get before below list
objects_in_view = aw.look()
print(" below list:", objects_in_view)


In [ ]:
command = """
Good. Now fly to the orchid. I want to inspect it from multiple angles, so orbit around it from 90 to -90 degrees at a radius of 2 meters, checking at 4 evenly spaced points. The drone should face the orchid at each point.
"""

ret = my_agent.process(command, False) # Don't execute
print("ret:\n", ret)

In [ ]:
import math
import time

# get position
orchid_position = aw.get_position('')

# defines half angle, endangle and half 
start_angle = 90
end_angle = -90
radius = 2

# calculate4 check angle
angles = [start_angle + i * (end_angle - start_angle) / 3 for i in range(4)]

# each angle
for angle in angles:
# angleconvertradians
rad_angle = math.radians(angle)
# calculatecheck position
check_x = orchid_position[0] + radius * math.cos(rad_angle)
check_y = orchid_position[1] + radius * math.sin(rad_angle)
check_z = orchid_position[2]
check_point = [check_x, check_y, check_z, angle]

# check
aw.fly_to(check_point)

# calculate from check angle
dx = orchid_position[0] - check_x
dy = orchid_position[1] - check_y
look_angle = math.atan2(dy, dx)
look_degrees = math.degrees(look_angle)

# let droneheading
aw.look_at(look_degrees)

# before below 
objects_in_view = aw.look()
print(f" in angle {angle}: {objects_in_view}")